# Relative probabilities of opening structures

A priori frequencies for one seat. The major-suit length buckets are mutually exclusive:
two-suiters are grouped under exactly 4 or exactly 5 cards in the major, while all hands
with 6+ cards in the major are grouped together regardless of a second long suit.

In [1]:
@file:DependsOn("com.github.phisgr:rektdeal:0.3.0")

import com.github.phisgr.dds.*
import com.github.phisgr.rektdeal.*

println("threads available: $threadCount")

fun report(labels: List<String>, counts: LongArray, nDeals: Int) {
    val totalHits = counts.sum()
    println("hits among $nDeals north hands: $totalHits\n")
    println("%-22s %8s %10s %12s".format("structure", "count", "per 1k", "share"))
    labels.forEachIndexed { i, label ->
        val c = counts[i]
        val perK = 1000.0 * c / nDeals
        val share = if (totalHits == 0L) 0.0 else 100.0 * c / totalHits
        println("%-22s %8d %10.2f %11.1f%%".format(label, c, perK, share))
    }
}

fun sample(nDeals: Int, categoryCount: Int, classify: (Hand) -> Int): LongArray {
    log("sampling $nDeals deals…")
    val threadCounts = multiThread(
        count = nDeals,
        state = { LongArray(categoryCount) },
        action = { dealCount, deal, counts ->
            if (dealCount % 1_000_000 == 0) log("$dealCount deals")
            val i = classify(deal.north)
            if (i >= 0) counts[i]++
        },
    )
    log("done")
    return LongArray(categoryCount) { i -> threadCounts.sumOf { it[i] } }
}


threads available: 12


## 1H opening

1. 10–15 HCP, exactly 4♥️, 5+♠️
2. 10–15 HCP, exactly 4♥️, 5+ minor
3. 10–15 HCP, exactly 5♥️, 5+ minor
4. 10–15 HCP, 6+♥️
5. 10–15 HCP, 4414 (singleton ♦)
6. 14–16 HCP, balanced 5♥️332
7. 14–16 HCP, balanced both majors 44(32)

In [2]:
val fourHeartFivePlusSpade = Shape { s, h, _, _ -> h == 4 && s >= 5 }
val fourHeartFivePlusMinor = Shape { _, h, d, c -> h == 4 && (d >= 5 || c >= 5) }
val fiveHeartFivePlusMinor = Shape { _, h, d, c -> h == 5 && (d >= 5 || c >= 5) }
val sixPlusHeart = Shape { _, h, _, _ -> h >= 6 }
val singletonDiamond4414 = Shape("4414")
val fiveHeart332 = Shape("(5332)").intersect(Shape { _, h, _, _ -> h == 5 })
val bothMajors4432 = Shape("44(32)")

val heartLabels = listOf(
    "10-15 4H 5+S",
    "10-15 4H 5+m",
    "10-15 5H 5+m",
    "10-15 6+H",
    "10-15 4414",
    "14-16 bal 5H332",
    "14-16 bal 44(32)",
)

fun classifyHeart(hand: Hand): Int = when {
    hand.hcp in 10..15 && fourHeartFivePlusSpade(hand) -> 0
    hand.hcp in 10..15 && fourHeartFivePlusMinor(hand) -> 1
    hand.hcp in 10..15 && fiveHeartFivePlusMinor(hand) -> 2
    hand.hcp in 10..15 && sixPlusHeart(hand) -> 3
    hand.hcp in 10..15 && singletonDiamond4414(hand) -> 4
    hand.hcp in 14..16 && fiveHeart332(hand) -> 5
    hand.hcp in 14..16 && bothMajors4432(hand) -> 6
    else -> -1
}

In [3]:
val nDeals = 10_000_000
val heartCounts = sample(nDeals, heartLabels.size, ::classifyHeart)

23:00:43 sampling 10000000 deals…


23:00:44 1000000 deals


23:00:44 2000000 deals


23:00:44 3000000 deals


23:00:44 4000000 deals


...

23:00:45 6000000 deals


23:00:45 7000000 deals


23:00:45 8000000 deals


23:00:45 9000000 deals


23:00:45 10000000 deals
23:00:45 done


## 1S opening

1. 10–15 HCP, exactly 5♠️ and 5+ in another suit
2. 10–15 HCP, exactly 4♠️ and 5+ in another suit
3. 10–15 HCP, 6+♠️ (including hands with a second 5+ suit)
4. 14–16 HCP, balanced 5♠️332

In [4]:
val fiveSpadeFivePlusSide = Shape { s, h, d, c -> s == 5 && maxOf(h, d, c) >= 5 }
val fourSpadeFivePlusSide = Shape { s, h, d, c -> s == 4 && maxOf(h, d, c) >= 5 }
val sixPlusSpade = Shape { s, _, _, _ -> s >= 6 }
val fiveSpade332 = Shape("5(332)")

val spadeLabels = listOf(
    "10-15 5S 5+x",
    "10-15 4S 5+x",
    "10-15 6+S",
    "14-16 bal 5S332",
)

fun classifySpade(hand: Hand): Int = when {
    hand.hcp in 10..15 && fiveSpadeFivePlusSide(hand) -> 0
    hand.hcp in 10..15 && fourSpadeFivePlusSide(hand) -> 1
    hand.hcp in 10..15 && sixPlusSpade(hand) -> 2
    hand.hcp in 14..16 && fiveSpade332(hand) -> 3
    else -> -1
}

val spadeCounts = sample(nDeals, spadeLabels.size, ::classifySpade)

23:00:46 sampling 10000000 deals…


23:00:46 1000000 deals


23:00:46 2000000 deals


23:00:46 3000000 deals


23:00:47 4000000 deals


...

23:00:47 6000000 deals


23:00:47 7000000 deals


23:00:47 8000000 deals


23:00:47 9000000 deals


23:00:48 10000000 deals
23:00:48 done


## Results

The samples above estimate each structure's absolute frequency and its share within the corresponding opening.

In [5]:
println("1H opening")
report(heartLabels, heartCounts, nDeals)
println("\n------------------------------\n")
println("1S opening")
report(spadeLabels, spadeCounts, nDeals)

1H opening
hits among 10000000 north hands: 793716



structure                 count     per 1k        share


10-15 4H 5+S             119935      11.99        15.1%


10-15 4H 5+m             239172      23.92        30.1%


10-15 5H 5+m              71576       7.16         9.0%


...

hits among 10000000 north hands: 749407

structure                 count     per 1k        share


10-15 5S 5+x             107449      10.74        14.3%


10-15 4S 5+x             360159      36.02        48.1%


10-15 6+S                229956      23.00        30.7%


14-16 bal 5S332           51843       5.18         6.9%


`share` is relative among that opening's structures only.
`per 1k` is absolute frequency per 1000 random north hands.